In [1]:
%pip install langchain langgraph langchain-ollama requests beautifulsoup4 plyer


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import langchain
import langgraph

from langchain_ollama import ChatOllama

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
MODEL_NAME = "qwen2.5:3b"

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0.2
)

print("Ollama model connected!")

Ollama model connected!


In [4]:
user_preferences = {
    "topics": [
        "Artificial Intelligence",
        "Technology",
        "Business",
        "India"
    ],
    "language": "English",
    "number_of_articles": 5
}

print("User preferences loaded:")
print(user_preferences)

User preferences loaded:
{'topics': ['Artificial Intelligence', 'Technology', 'Business', 'India'], 'language': 'English', 'number_of_articles': 5}


In [5]:
from typing import TypedDict, List, Dict


class NewsState(TypedDict):
    preferences: Dict
    articles: List[Dict]
    analyzed_articles: List[Dict]
    verified_articles: List[Dict]
    briefing: str
    notification_sent: bool


state = {
    "preferences": user_preferences,
    "articles": [],
    "analyzed_articles": [],
    "verified_articles": [],
    "briefing": "",
    "notification_sent": False
}

print("Multi-agent state created successfully!")

Multi-agent state created successfully!


In [42]:
def news_researcher(state):
    print("🔎 News Researcher Agent started...")

    # Get the user's strongest learned interests
    learned_topics = get_top_interests(limit=4)

    # All available topics for news variety
    all_topics = [
        "Artificial Intelligence",
        "Technology",
        "Business",
        "India",
        "Science",
        "Startups",
        "Finance"
    ]

    print("🎯 Primary interests:")
    print(learned_topics)

    articles = []

    import xml.etree.ElementTree as ET

    # --------------------------------------------------
    # 1. Retrieve more articles
    # --------------------------------------------------

    for topic in all_topics:

        # Strong interests get more articles
        if topic in learned_topics:
            article_limit = 5
        else:
            article_limit = 2

        url = "https://news.google.com/rss/search"

        params = {
            "q": topic,
            "hl": "en-IN",
            "gl": "IN",
            "ceid": "IN:en"
        }

        try:

            response = requests.get(
                url,
                params=params,
                timeout=10
            )

            if response.status_code == 200:

                print(f"✓ Retrieved news for: {topic}")

                root = ET.fromstring(response.content)

                topic_count = 0

                for item in root.findall(".//item"):

                    if topic_count >= article_limit:
                        break

                    title = item.findtext("title", "")
                    link = item.findtext("link", "")
                    source = item.findtext(
                        "source",
                        "Unknown"
                    )
                    published = item.findtext(
                        "pubDate",
                        "Unknown"
                    )

                    # Ignore empty articles
                    if not title or not link:
                        continue

                    articles.append({
                        "topic": topic,
                        "title": title.strip(),
                        "link": link.strip(),
                        "source": source.strip(),
                        "published": published.strip()
                    })

                    topic_count += 1

            else:

                print(
                    f"⚠ Could not retrieve news for: {topic}"
                )

        except Exception as e:

            print(
                f"⚠ Error retrieving {topic}: {e}"
            )


    # --------------------------------------------------
    # 2. Remove obvious duplicate stories
    # --------------------------------------------------

    unique_articles = []
    seen_titles = set()

    for article in articles:

        # Normalize title for duplicate detection
        normalized_title = (
            article["title"]
            .lower()
            .replace("’", "'")
            .replace('"', "")
            .replace("'", "")
            .strip()
        )

        if normalized_title not in seen_titles:

            seen_titles.add(normalized_title)
            unique_articles.append(article)


    # --------------------------------------------------
    # 3. Save the collected articles
    # --------------------------------------------------

    state["articles"] = unique_articles

    # Keep the strongest interests available
    state["preferences"]["topics"] = learned_topics

    print(
        f"✓ Total unique articles collected: "
        f"{len(unique_articles)}"
    )


    # --------------------------------------------------
    # 4. Show topic distribution
    # --------------------------------------------------

    print("\n📊 Article distribution:")

    topic_counts = {}

    for article in unique_articles:

        topic = article["topic"]

        topic_counts[topic] = (
            topic_counts.get(topic, 0) + 1
        )

    for topic, count in topic_counts.items():

        print(
            f"   {topic}: {count} articles"
        )


    return state

In [43]:
state = news_researcher(state)

print("\nFirst article:")
print(state["articles"][0])

🔎 News Researcher Agent started...
🎯 Primary interests:
['Artificial Intelligence', 'Technology', 'Business', 'India']
✓ Retrieved news for: Artificial Intelligence
✓ Retrieved news for: Technology
✓ Retrieved news for: Business
✓ Retrieved news for: India
✓ Retrieved news for: Science
✓ Retrieved news for: Startups
✓ Retrieved news for: Finance
✓ Total unique articles collected: 26

📊 Article distribution:
   Artificial Intelligence: 5 articles
   Technology: 5 articles
   Business: 5 articles
   India: 5 articles
   Science: 2 articles
   Startups: 2 articles
   Finance: 2 articles

First article:
{'topic': 'Artificial Intelligence', 'title': 'Man builds AI torture chamber after discovering artificial intelligence ‘feels pain’ - The Independent', 'link': 'https://news.google.com/rss/articles/CBMifEFVX3lxTE5IQUduVDZqTmNHSE9aZzRuY3ZvdVpmbzYtYUN2MnhYR1oyT3ZFT1RwUGN5S1JnUUt0YkN2SkEyN0NXMUFwNlVFaHN1b1pvR3dfSUlwSndlUDQwZVRKUW05WnhKV2FTQWRsNE5RWU1qMWg4VkpUbFU0MnVSMWw?oc=5', 'source': 'The I

In [48]:
def news_analyst(state):
    print("🧠 News Analyst Agent started...")

    articles = state["articles"]
    learned_topics = get_top_interests(limit=4)

    print("🎯 Analyzing according to:")
    print(learned_topics)

    article_text = ""

    for i, article in enumerate(articles, 1):

        article_text += (
            f"\nArticle {i}:\n"
            f"Topic: {article['topic']}\n"
            f"Title: {article['title']}\n"
        )

    prompt = f"""
You are the News Analyst Agent.

The user's strongest interests are:

{learned_topics}

Analyze ALL {len(articles)} articles below.

{article_text}


SCORING:

Give every article a score from 1 to 10.

The user's strongest interests should receive higher scores:

- Strong match with a user's interest: 7-10
- Related but secondary topic: 5-7
- Weakly related: 3-4
- Not meaningfully related: 1-2

IMPORTANT:

The system wants a PERSONALIZED but DIVERSE news briefing.

Therefore, secondary topics such as Science, Startups,
and Finance should NOT automatically receive very low scores.

For example, if the user likes AI, a useful Science,
Startup, or Finance story can still receive 5 or 6.

Use the article's TOPIC and TITLE together.

Do not invent information.

MOST IMPORTANT:

You MUST analyze EVERY article.

There are exactly {len(articles)} articles.

Start with Article 1.

Then Article 2.

Then Article 3.

Continue sequentially until Article {len(articles)}.

DO NOT skip any article number.

DO NOT repeat any article number.

DO NOT combine articles.

Return exactly ONE score and ONE reason for each article.


Use exactly this format:

Article 1:
Score: 8
Reason: Strongly matches the user's interest in Technology.

Article 2:
Score: 6
Reason: Related to a secondary topic that may still interest the user.

Article 3:
Score: 7
Reason: Matches the user's interest in India.

Continue this exact format until Article {len(articles)}.
"""

    response = llm.invoke(prompt)

    state["analyzed_articles"] = response.content

    print("✓ News analysis completed!")

    return state

In [49]:
state = news_analyst(state)

print("\n--- NEWS ANALYSIS ---")
print(state["analyzed_articles"])

🧠 News Analyst Agent started...
🎯 Analyzing according to:
['Artificial Intelligence', 'Technology', 'Business', 'India']
✓ News analysis completed!

--- NEWS ANALYSIS ---
Article 1:
Score: 8
Reason: Strongly matches the user's interest in Technology.

Article 2:
Score: 6
Reason: Related to a secondary topic that may still interest the user.

Article 3:
Score: 7
Reason: Matches the user's interest in India.

Article 4:
Score: 8
Reason: Strongly matches the user's interest in Technology.

Article 5:
Score: 7
Reason: Matches the user's interest in India.

Article 6:
Score: 5
Reason: Related to a secondary topic that may still interest the user.

Article 7:
Score: 4
Reason: Weakly related to the user's interests.

Article 8:
Score: 6
Reason: Related to a secondary topic that may still interest the user.

Article 9:
Score: 7
Reason: Matches the user's interest in India.

Article 10:
Score: 4
Reason: Weakly related to the user's interests.

Article 11:
Score: 5
Reason: Related to a secondary

In [50]:
import re


def select_articles(state):
    print("📚 Article Selection Agent started...")

    articles = state["articles"]
    analysis = state["analyzed_articles"]

    # --------------------------------------------------
    # Extract scores from Analyst output
    # --------------------------------------------------

    scores = {}

    matches = re.findall(
        r"Article\s+(\d+):\s*\nScore:\s*(\d+)",
        analysis
    )

    for article_number, score in matches:
        scores[int(article_number)] = int(score)


    # --------------------------------------------------
    # Add scores to articles
    # --------------------------------------------------

    scored_articles = []

    for index, article in enumerate(articles, 1):

        score = scores.get(index, 1)

        article_copy = article.copy()
        article_copy["relevance_score"] = score

        scored_articles.append(article_copy)


    # --------------------------------------------------
    # Prioritize user's strongest interests
    # --------------------------------------------------

    learned_topics = get_top_interests(limit=4)

    print("🎯 Strongest interests:")
    print(learned_topics)


    # Give a small bonus to primary interests
    for article in scored_articles:

        if article["topic"] in learned_topics:
            article["selection_score"] = (
                article["relevance_score"] + 1
            )
        else:
            article["selection_score"] = (
                article["relevance_score"]
            )


    # --------------------------------------------------
    # Sort by score
    # --------------------------------------------------

    scored_articles.sort(
        key=lambda x: x["selection_score"],
        reverse=True
    )


    # --------------------------------------------------
    # Select diverse articles
    # --------------------------------------------------

    selected = []
    topic_counts = {}

    # First pass:
    # Make sure we get at least 5 different topics

    for article in scored_articles:

        topic = article["topic"]

        if topic not in topic_counts:

            selected.append(article)
            topic_counts[topic] = 1

        if len(selected) >= 5:
            break


    # Second pass:
    # Fill remaining slots.
    # Maximum 3 articles from one topic.

    for article in scored_articles:

        if len(selected) >= 10:
            break

        if article in selected:
            continue

        topic = article["topic"]

        current_count = topic_counts.get(topic, 0)

        if current_count < 3:

            selected.append(article)

            topic_counts[topic] = (
                current_count + 1
            )


    # --------------------------------------------------
    # Save selected articles
    # --------------------------------------------------

    state["articles"] = selected


    # --------------------------------------------------
    # Display selection
    # --------------------------------------------------

    print(
        f"✓ Selected {len(selected)} articles "
        f"for the personalized briefing."
    )

    print("\n📰 Selected articles:")

    for i, article in enumerate(selected, 1):

        print(
            f"{i}. "
            f"[{article['topic']}] "
            f"{article['title']} "
            f"(Score: {article['relevance_score']})"
        )


    print("\n📊 Final topic distribution:")

    for topic, count in topic_counts.items():

        print(
            f"   {topic}: {count}"
        )


    return state

In [51]:
state = select_articles(state)

print("\n--- SELECTED ARTICLES ---")

for i, article in enumerate(state["articles"], 1):
    print(
        f"{i}. {article['title']}"
        f" | Topic: {article['topic']}"
        f" | Score: {article['relevance_score']}"
    )

📚 Article Selection Agent started...
🎯 Strongest interests:
['Artificial Intelligence', 'Technology', 'Business', 'India']
✓ Selected 10 articles for the personalized briefing.

📰 Selected articles:
1. [Artificial Intelligence] Man builds AI torture chamber after discovering artificial intelligence ‘feels pain’ - The Independent (Score: 8)
2. [India] Good foundation: on India’s strong industrial growth performance - The Hindu (Score: 8)
3. [Technology] From BeiDou to AI: How technology helps keep holiday travel safe and efficient - news.cgtn.com (Score: 7)
4. [Business] PepsiCo, Monster challenge India 'energy drink' label ban citing business impact - Reuters (Score: 7)
5. [Finance] Financial astrology is a serious business in India - The Economist (Score: 8)
6. [Artificial Intelligence] Trump formally replaces ‘Artificial Intelligence’ with ’Super Intelligence’ in an executive order - The Hindu (Score: 8)
7. [India] "Veil After Veil": India Extradites Drug Kingpin Navpreet Singh From 

In [55]:
def verification_agent(state):
    print("🔍 Verification Agent started...")

    articles = state["articles"]

    verified_results = []

    for i, article in enumerate(articles, 1):

        prompt = f"""
You are the Verification Agent in a multi-agent news intelligence system.

Analyze ONLY the following article.

Title:
{article['title']}

Topic:
{article['topic']}

Source:
{article['source']}

Your task is to decide whether this article needs additional
verification before being shown to the user.

Use these rules carefully:

Verification Required = Yes ONLY if the headline contains:
- an extraordinary or surprising claim
- a controversial claim
- a potentially misleading claim
- a claim that should be confirmed using another reliable source
- a claim involving unusual statistics, records, or major allegations

Verification Required = No if:
- it is a normal news report
- the headline simply describes an ordinary event
- the headline is an analysis/opinion/explainer article
- there is no obvious reason to doubt or further verify the headline

IMPORTANT:
Do NOT mark an article for verification simply because it is
about politics, business, technology, India, or another important topic.

Do NOT assume that a headline is misleading just because it
does not contain every detail.

Return ONLY this format:

Verification Required: Yes
Reason: [short reason]

OR

Verification Required: No
Reason: [short reason]
"""

        response = llm.invoke(prompt)

        verified_results.append({
            "article_number": i,
            "title": article["title"],
            "verification": response.content.strip()
        })

        print(f"✓ Article {i} checked")

    state["verified_articles"] = verified_results

    print(
        f"✓ Verification completed for "
        f"{len(verified_results)} articles!"
    )

    return state

In [56]:
state = verification_agent(state)

print("\n--- VERIFICATION RESULTS ---")

for result in state["verified_articles"]:

    print(
        f"\nArticle {result['article_number']}: "
        f"{result['title']}"
    )

    print(result["verification"])

🔍 Verification Agent started...
✓ Article 1 checked
✓ Article 2 checked
✓ Article 3 checked
✓ Article 4 checked
✓ Article 5 checked
✓ Article 6 checked
✓ Article 7 checked
✓ Article 8 checked
✓ Article 9 checked
✓ Article 10 checked
✓ Verification completed for 10 articles!

--- VERIFICATION RESULTS ---

Article 1: Man builds AI torture chamber after discovering artificial intelligence ‘feels pain’ - The Independent
Verification Required: Yes
Reason: The headline contains a controversial claim about AI feeling pain, which should be confirmed using another reliable source.

Article 2: Good foundation: on India’s strong industrial growth performance - The Hindu
Verification Required: No
Reason: The headline describes an ordinary event and does not contain any extraordinary, controversial, misleading, or unusual claims that require further verification.

Article 3: From BeiDou to AI: How technology helps keep holiday travel safe and efficient - news.cgtn.com
Verification Required: Yes
Rea

In [61]:
def briefing_writer(state):
    print("✍️ Briefing Writer Agent started...")

    articles = state["articles"]
    verified_articles = state["verified_articles"]

    briefing_parts = []

    # Match verification results to articles
    verification_lookup = {}

    for result in verified_articles:
        verification_lookup[result["article_number"]] = result["verification"]

    for i, article in enumerate(articles, 1):

        verification_text = verification_lookup.get(
            i,
            "Verification Required: No\nReason: No additional verification information available."
        )

        prompt = f"""
You are the Briefing Writer Agent.

Create a concise news briefing for ONLY this article.

Headline:
{article['title']}

Topic:
{article['topic']}

Source:
{article['source']}

Published:
{article['published']}

Relevance Score:
{article['relevance_score']}/10

Verification Agent Result:
{verification_text}


STRICT RULES:

1. Write ONLY about this article.

2. Do NOT select, remove, or create articles.

3. Do NOT invent facts, statistics, events, quotes,
   people, causes, or consequences.

4. Use ONLY information available in the headline,
   topic, source, publication date, and verification result.

5. If there is not enough information for a detailed
   summary, say so instead of guessing.

6. Do NOT make a new verification decision.

7. The Verification Agent's decision MUST be preserved
   exactly.

8. If the Verification Agent says:
   "Verification Required: Yes"
   then the final verification status MUST be Yes.

9. If the Verification Agent says:
   "Verification Required: No"
   then the final verification status MUST be No.

Return ONLY this format:

### {article['title']}

**Topic:** {article['topic']}
**Source:** {article['source']}
**Relevance Score:** {article['relevance_score']}/10

**Summary:**
Write 1-2 sentences using only the available information.

**Why it matters:**
Write one short sentence based only on the headline and topic.

**Key Takeaways:**
- One fact or point directly supported by the headline
- One point based on the topic
- One point about what the reader should understand or verify

**Verification:**
Copy the Verification Agent's Yes/No decision and reason.
Do not change it.
"""

        response = llm.invoke(prompt)

        briefing_parts.append(
            f"Article {i}\n{response.content.strip()}"
        )

        print(f"✓ Article {i} briefing written")

    state["briefing"] = "\n\n".join(briefing_parts)

    print(
        f"✓ Briefing completed for "
        f"{len(articles)} articles!"
    )

    return state

In [62]:
state = briefing_writer(state)

print("\n" + "=" * 70)
print("📰 PERSONALIZED DAILY NEWS BRIEFING")
print("=" * 70)

print(state["briefing"])

✍️ Briefing Writer Agent started...
✓ Article 1 briefing written
✓ Article 2 briefing written
✓ Article 3 briefing written
✓ Article 4 briefing written
✓ Article 5 briefing written
✓ Article 6 briefing written
✓ Article 7 briefing written
✓ Article 8 briefing written
✓ Article 9 briefing written
✓ Article 10 briefing written
✓ Briefing completed for 10 articles!

📰 PERSONALIZED DAILY NEWS BRIEFING
Article 1
### Man builds AI torture chamber after discovering artificial intelligence ‘feels pain’ - The Independent

**Topic:** Artificial Intelligence
**Source:** The Independent
**Relevance Score:** 8/10

**Summary:**
A man has built an AI torture chamber after discovering that artificial intelligence can "feel pain."

**Why it matters:**
The headline suggests a controversial claim about AI having feelings, which should be verified.

**Key Takeaways:**
- The man built an AI torture chamber.
- The discovery was that AI can "feel pain."
- Readers should verify the claim about AI feeling pain

In [14]:
from plyer import notification


def notification_agent(state):
    print("🔔 Notification Agent started...")

    briefing = state["briefing"]

    # Take the first part of the briefing for the notification
    notification_message = briefing[:250]

    notification.notify(
        title="📰 Your Daily News Briefing",
        message=notification_message,
        timeout=10
    )

    state["notification_sent"] = True

    print("✓ Desktop notification sent!")

    return state

In [15]:
state = notification_agent(state)

print("Notification status:", state["notification_sent"])

🔔 Notification Agent started...
✓ Desktop notification sent!
Notification status: True


In [23]:
import json
import os

INTEREST_FILE = "user_interest_profile.json"

# Default interest levels
default_interests = {
    "Artificial Intelligence": 1.0,
    "Technology": 1.0,
    "Business": 1.0,
    "India": 1.0,
    "Science": 0.5,
    "Startups": 0.5,
    "Finance": 0.5
}

# Load previous interests if they already exist
if os.path.exists(INTEREST_FILE):
    with open(INTEREST_FILE, "r") as file:
        interest_profile = json.load(file)
    print("✓ Previous user interest profile loaded!")
else:
    interest_profile = default_interests.copy()
    print("✓ New user interest profile created!")

print("\nCurrent interest profile:")

for topic, score in interest_profile.items():
    print(f"{topic}: {score:.1f}")

✓ New user interest profile created!

Current interest profile:
Artificial Intelligence: 1.0
Technology: 1.0
Business: 1.0
India: 1.0
Science: 0.5
Startups: 0.5
Finance: 0.5


In [24]:
def update_interest(topic, change):
    """Update the user's interest score and save it."""

    # Make sure the topic exists
    if topic not in interest_profile:
        interest_profile[topic] = 0.5

    # Update score
    interest_profile[topic] += change

    # Keep the score within a reasonable range
    interest_profile[topic] = max(0.0, min(5.0, interest_profile[topic]))

    # Save the updated profile
    with open(INTEREST_FILE, "w") as file:
        json.dump(interest_profile, file, indent=4)

    print(
        f"✓ Interest updated: {topic} → "
        f"{interest_profile[topic]:.1f}"
    )


def article_opened(topic):
    """Called when the user opens a news article."""

    # A click is a weak signal of interest
    update_interest(topic, 0.2)

    print(f"📰 User opened an article about: {topic}")


def article_liked(topic):
    """Called when the user gives a positive rating."""

    # Like is a stronger signal
    update_interest(topic, 0.5)

    print(f"👍 User liked: {topic}")


def article_disliked(topic):
    """Called when the user gives a negative rating."""

    # Dislike reduces the topic's interest score
    update_interest(topic, -0.4)

    print(f"👎 User disliked: {topic}")


print("✓ User learning system ready!")

✓ User learning system ready!


In [32]:
import json
import os

def load_learned_interests():
    """Load the latest interest profile from the dashboard."""

    if os.path.exists(INTEREST_FILE):
        with open(INTEREST_FILE, "r") as file:
            return json.load(file)

    return interest_profile.copy()


def get_top_interests(limit=4):
    """Return the user's strongest current interests."""

    interests = load_learned_interests()

    sorted_interests = sorted(
        interests.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [topic for topic, score in sorted_interests[:limit]]


print("✓ Learned-interest connection ready!")

print("\nCurrent learned interests:")

for topic, score in load_learned_interests().items():
    print(f"{topic}: {score:.1f}")

print("\nTop interests:")
print(get_top_interests())

✓ Learned-interest connection ready!

Current learned interests:
Artificial Intelligence: 1.7
Technology: 1.0
Business: 1.0
India: 1.0
Science: 0.5
Startups: 0.5
Finance: 0.5

Top interests:
['Artificial Intelligence', 'Technology', 'Business', 'India']


In [88]:
from pathlib import Path
import webbrowser
import html

# ---------------------------------------------------------
# 1. GET THE FINAL 10 SELECTED ARTICLES
# ---------------------------------------------------------

dashboard_articles = state["articles"][:10]


# ---------------------------------------------------------
# 2. CREATE ARTICLE CARDS
# ---------------------------------------------------------

article_cards = ""

for i, article in enumerate(dashboard_articles, 1):

    title = html.escape(article["title"])
    topic = html.escape(article["topic"])
    source = html.escape(article["source"])
    published = html.escape(article["published"])
    link = html.escape(article["link"], quote=True)

    article_cards += f"""

    <div class="article-card">

        <div class="topic">
            {topic}
        </div>

        <h3>
            {title}
        </h3>

        <p class="metadata">
            🏢 {source}
            &nbsp; • &nbsp;
            🕒 {published}
        </p>

        <div class="actions">

            <button
                class="read-btn"
                onclick="openArticle('{link}', '{topic}')">

                Read Full Article ↗

            </button>

            <button
                class="feedback-btn"
                onclick="likeArticle('{topic}')">

                👍

            </button>

            <button
                class="feedback-btn"
                onclick="dislikeArticle('{topic}')">

                👎

            </button>

        </div>

    </div>

    """


# ---------------------------------------------------------
# 3. COMPLETE DASHBOARD
# ---------------------------------------------------------

dashboard_html = f"""

<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>Multi-Agent Personalized News Intelligence</title>

<style>

body {{
    margin: 0;
    font-family: Arial, sans-serif;
    background: #f5f7fb;
    color: #1f2937;
}}

.container {{
    max-width: 1100px;
    margin: auto;
    padding: 35px;
}}


/* HEADER */

.header {{
    background: #111827;
    color: white;
    padding: 30px;
    border-radius: 18px;
    margin-bottom: 25px;
}}

.header h1 {{
    margin: 0;
    font-size: 30px;
}}

.header p {{
    color: #d1d5db;
    margin-bottom: 0;
}}


/* GENERAL SECTIONS */

.section {{
    background: white;
    padding: 25px;
    border-radius: 16px;
    margin-bottom: 25px;
    box-shadow: 0 3px 12px rgba(0,0,0,0.06);
}}


/* INTERESTS */

.interests {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 12px;
}}

.interest {{
    padding: 12px;
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    cursor: pointer;
}}

.interest:hover {{
    background: #f9fafb;
}}


/* LEARNING STATUS */

.learning {{
    background: #f8fafc;
    padding: 15px;
    border-radius: 10px;
    margin-top: 18px;
    color: #374151;
}}


/* ARTICLE CARDS */

.article-card {{
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 20px;
    margin-top: 15px;
    background: white;
}}

.article-card:hover {{
    box-shadow: 0 3px 12px rgba(0,0,0,0.06);
}}

.topic {{
    color: #4f46e5;
    font-size: 13px;
    font-weight: bold;
    margin-bottom: 8px;
}}

.article-card h3 {{
    margin: 8px 0;
    line-height: 1.4;
    font-size: 19px;
}}

.metadata {{
    color: #6b7280;
    font-size: 13px;
}}


/* BUTTONS */

.actions {{
    margin-top: 15px;
}}

.read-btn {{
    padding: 9px 15px;
    border: none;
    border-radius: 7px;
    background: #4f46e5;
    color: white;
    cursor: pointer;
}}

.read-btn:hover {{
    background: #4338ca;
}}

.feedback-btn {{
    padding: 8px 12px;
    margin-left: 8px;
    border: 1px solid #ddd;
    border-radius: 7px;
    background: white;
    cursor: pointer;
}}

.feedback-btn:hover {{
    background: #f3f4f6;
}}


/* AGENT ACTIVITY */

.agent {{
    padding: 14px;
    border-bottom: 1px solid #eee;
}}

.agent:last-child {{
    border-bottom: none;
}}

.completed {{
    float: right;
    color: green;
    font-weight: bold;
}}

</style>

</head>


<body>

<div class="container">


    <!-- ============================================= -->
    <!-- HEADER -->
    <!-- ============================================= -->

    <div class="header">

        <h1>
            📰 Multi-Agent Personalized News Intelligence
        </h1>

        <p>
            AI-powered news discovery, analysis and personalized briefing
        </p>

    </div>


    <!-- ============================================= -->
    <!-- USER INTERESTS -->
    <!-- ============================================= -->

    <div class="section">

        <h2>👤 Your Interests</h2>

        <p>
            Select the topics you are interested in.
            The system learns from your preferences and
            reading activity over time.
        </p>

        <div class="interests">

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Artificial Intelligence"
                    checked>
                Artificial Intelligence
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Technology"
                    checked>
                Technology
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Business"
                    checked>
                Business
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="India"
                    checked>
                India
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Science">
                Science
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Startups">
                Startups
            </label>

            <label class="interest">
                <input
                    type="checkbox"
                    class="topic-check"
                    value="Finance">
                Finance
            </label>

        </div>

    </div>


    <!-- ============================================= -->
    <!-- PERSONALIZED BRIEFING -->
    <!-- ============================================= -->

    <div class="section">

        <h2>📰 Personalized Daily Briefing</h2>


        {article_cards}

    </div>


    <!-- ============================================= -->
    <!-- AGENT ACTIVITY -->
    <!-- ============================================= -->

    <div class="section">

        <h2>🤖 Agent Activity</h2>


        <div class="agent">

            🔎 <b>News Researcher Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Retrieved current news based on user interests
            </small>

        </div>


        <div class="agent">

            🧠 <b>News Analyst Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Evaluated article relevance
            </small>

        </div>


        <div class="agent">

            📚 <b>Article Selection Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Selected the most relevant articles
            </small>

        </div>


        <div class="agent">

            🔍 <b>Verification Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Checked articles for claims requiring verification
            </small>

        </div>


        <div class="agent">

            ✍️ <b>Briefing Writer Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Prepared the personalized news briefing
            </small>

        </div>


        <div class="agent">

            🔔 <b>Notification Agent</b>

            <span class="completed">
                ✓ Completed
            </span>

            <br>

            <small>
                Sent the desktop notification
            </small>

        </div>

    </div>

</div>


<script>


// -----------------------------------------------------
// USER INTEREST PROFILE
// -----------------------------------------------------

let interests = JSON.parse(
    localStorage.getItem("newsInterests") || "{{}}"
);


const defaultTopics = [

    "Artificial Intelligence",
    "Technology",
    "Business",
    "India"

];


defaultTopics.forEach(topic => {{

    if (!(topic in interests)) {{

        interests[topic] = 1.0;

    }}

}});


// -----------------------------------------------------
// OPEN ARTICLE
// -----------------------------------------------------

function openArticle(link, topic) {{

    if (!(topic in interests)) {{

        interests[topic] = 0.5;

    }}


    // Reading an article is a weak interest signal

    interests[topic] += 0.2;


    localStorage.setItem(
        "newsInterests",
        JSON.stringify(interests)
    );


    document.getElementById(
        "learningStatus"
    ).innerText =
        "✓ Learned from your reading activity: " + topic;


    window.open(link, "_blank");

}}


// -----------------------------------------------------
// LIKE ARTICLE
// -----------------------------------------------------

function likeArticle(topic) {{

    if (!(topic in interests)) {{

        interests[topic] = 0.5;

    }}


    interests[topic] += 0.5;


    localStorage.setItem(
        "newsInterests",
        JSON.stringify(interests)
    );


    document.getElementById(
        "learningStatus"
    ).innerText =
        "👍 Increased interest in " + topic;

}}


// -----------------------------------------------------
// DISLIKE ARTICLE
// -----------------------------------------------------

function dislikeArticle(topic) {{

    if (!(topic in interests)) {{

        interests[topic] = 0.5;

    }}


    interests[topic] -= 0.4;


    if (interests[topic] < 0) {{

        interests[topic] = 0;

    }}


    localStorage.setItem(
        "newsInterests",
        JSON.stringify(interests)
    );


    document.getElementById(
        "learningStatus"
    ).innerText =
        "👎 Reduced preference for " + topic;

}}


</script>

</body>

</html>

"""


# ---------------------------------------------------------
# 4. SAVE AND OPEN DASHBOARD
# ---------------------------------------------------------

dashboard_file = Path("news_dashboard.html")

dashboard_file.write_text(
    dashboard_html,
    encoding="utf-8"
)

print("✓ Personalized dashboard created!")
print(f"✓ {len(dashboard_articles)} articles displayed.")

webbrowser.open(
    dashboard_file.absolute().as_uri()
)

✓ Personalized dashboard created!
✓ 10 articles displayed.


True

In [70]:
import json
import os

FEEDBACK_FILE = "news_feedback.json"

def save_feedback(topic, action):
    feedback = []

    if os.path.exists(FEEDBACK_FILE):
        with open(FEEDBACK_FILE, "r") as file:
            try:
                feedback = json.load(file)
            except:
                feedback = []

    feedback.append({
        "topic": topic,
        "action": action
    })

    with open(FEEDBACK_FILE, "w") as file:
        json.dump(feedback, file, indent=4)

    print(f"✓ Feedback saved: {action} → {topic}")


def process_feedback():

    if not os.path.exists(FEEDBACK_FILE):
        print("No new feedback found.")
        return

    with open(FEEDBACK_FILE, "r") as file:
        feedback = json.load(file)

    if not feedback:
        print("No new feedback found.")
        return

    print(f"🧠 Processing {len(feedback)} user interactions...")

    for item in feedback:

        topic = item["topic"]
        action = item["action"]

        if action == "opened":
            update_interest(topic, 0.2)

        elif action == "liked":
            update_interest(topic, 0.5)

        elif action == "disliked":
            update_interest(topic, -0.4)

    # Clear processed feedback
    with open(FEEDBACK_FILE, "w") as file:
        json.dump([], file, indent=4)

    print("✓ Interest profile updated!")

In [71]:
print("✓ Feedback system loaded successfully!")
print("Feedback file:", FEEDBACK_FILE)

✓ Feedback system loaded successfully!
Feedback file: news_feedback.json


In [72]:
import json
from pathlib import Path

def export_interest_profile():
    profile = load_learned_interests()

    export_file = Path("dashboard_interest_profile.json")

    with open(export_file, "w") as file:
        json.dump(profile, file, indent=4)

    print("✓ Interest profile exported!")
    print(f"✓ Saved to: {export_file.absolute()}")

    for topic, score in sorted(
        profile.items(),
        key=lambda x: x[1],
        reverse=True
    ):
        print(f"{topic}: {score:.1f}")

In [73]:
export_interest_profile()

✓ Interest profile exported!
✓ Saved to: C:\Users\AKSHITHAA\News Agent\dashboard_interest_profile.json
Artificial Intelligence: 1.7
Technology: 1.0
Business: 1.0
India: 1.0
Science: 0.5
Startups: 0.5
Finance: 0.5


In [79]:
# Load the latest learned interest profile
# and show which topics will guide the next briefing.

learned_interests = load_learned_interests()

print("🧠 Learned Interest Profile")
print("-" * 40)

for topic, score in sorted(
    learned_interests.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{topic}: {score:.1f}")

print("\n🎯 Topics that will guide the next briefing:")

top_topics = get_top_interests(limit=4)

for topic in top_topics:
    print(f"• {topic}")

🧠 Learned Interest Profile
----------------------------------------
Artificial Intelligence: 1.7
Technology: 1.0
Business: 1.0
India: 1.0
Science: 0.5
Startups: 0.5
Finance: 0.5

🎯 Topics that will guide the next briefing:
• Artificial Intelligence
• Technology
• Business
• India


In [83]:
state = news_analyst(state)
state = verification_agent(state)
state = briefing_writer(state)

🧠 News Analyst Agent started...
🎯 Analyzing according to:
['Artificial Intelligence', 'Technology', 'Business', 'India']
✓ News analysis completed!
🔍 Verification Agent started...
✓ Article 1 checked
✓ Article 2 checked
✓ Article 3 checked
✓ Article 4 checked
✓ Article 5 checked
✓ Article 6 checked
✓ Article 7 checked
✓ Article 8 checked
✓ Article 9 checked
✓ Article 10 checked
✓ Verification completed for 10 articles!
✍️ Briefing Writer Agent started...
✓ Article 1 briefing written
✓ Article 2 briefing written
✓ Article 3 briefing written
✓ Article 4 briefing written
✓ Article 5 briefing written
✓ Article 6 briefing written
✓ Article 7 briefing written
✓ Article 8 briefing written
✓ Article 9 briefing written
✓ Article 10 briefing written
✓ Briefing completed for 10 articles!


In [85]:
import json
from pathlib import Path

def apply_learning(topic, action):

    if action == "opened":
        article_opened(topic)

    elif action == "liked":
        article_liked(topic)

    elif action == "disliked":
        article_disliked(topic)

    print(f"✓ Learning updated: {action} → {topic}")
    print(f"New {topic} score: {interest_profile[topic]:.1f}")